# SOP for reading FASTQ Files from NGS Sequencing

1. Download the FASTQ files
2. Place them all in their own directory
3. Run the program

## Setup

These packages need to be installed and imported (using `pip`)

In [ ]:
%pip install -q numpy pandas paramiko

In [ ]:
import os, csv, glob, re, gzip, shutil
import numpy as np
import pandas as pd
import paramiko
from stat import S_ISDIR, S_ISREG

class Config():
    PATH = "./FASTQ" # update with your path to file or single FASTQ file
    TRANSLATE_FLAG = True
    MOTIF = None # if searching for motif
    FLANKED_BY = ["LVK", "LVY"] # if searching for sequence flanked by specified AA sequence

config = Config()

## Download Files from SFTP Server

Hostname, username, or password may need to be updated 

In [ ]:
import configparser
credentials = configparser.ConfigParser()
credentials.read('../credentials.ini')

# Define SFTP connection parameters
hostname = credentials['genwiz']['hostname']  # Replace with your SFTP server hostname or IP
username = credentials['genwiz']['username']  # Replace with your SFTP username
password = credentials['genwiz']['password']  # Replace with your SFTP password
port = 22  # Default SFTP port is 22

print(hostname, username, password)

root_dir = "genewiz-us-ngs-sftp"

# Create an SSH client
ssh_client = paramiko.SSHClient()
ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy()) # Or use WarningPolicy or RejectPolicy for stricter security

def download_dir_recursive(sftp, remote_path, local_path):
    os.makedirs(local_path, exist_ok=True)
    for entry in sftp.listdir_attr(remote_path):
        remote_item_path = f"{remote_path}/{entry.filename}"
        local_item_path = os.path.join(local_path, entry.filename)

        if S_ISDIR(entry.st_mode):
            download_dir_recursive(sftp, remote_item_path, local_item_path)
        elif S_ISREG(entry.st_mode):
            sftp.get(remote_item_path, local_item_path)
        else:
            print(f"Skipping unknown file type: {remote_item_path}")

# Connect to the SFTP server
try:
    ssh_client.connect(hostname, port, username, password)
    print(f"Connected to SFTP server: {hostname}")

    # Create an SFTP session
    sftp = ssh_client.open_sftp()
    print("SFTP session opened.")

    download_dir_recursive(sftp, f"/{root_dir}/{username}/", config.PATH)

except paramiko.AuthenticationException:
    print("Authentication failed. Check your username and password.")
except paramiko.SSHException as e:
    print(f"SSH connection failed: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Close the SFTP session and SSH client
    if 'sftp' in locals() and sftp:
        sftp.close()
        print("SFTP session closed.")
    if 'ssh_client' in locals() and ssh_client:
        ssh_client.close()
        print("SSH client closed.")

for file in glob.glob(os.path.join(config.PATH, "**", "*", "*.gz"), recursive=True):
    print(file)
    # Open and decompress the gzipped file
    with gzip.open(file, 'rb') as f_in:
        print(f"Unzipped {file}")
        with open(file[:-3], 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

for file in glob.glob(os.path.join(config.PATH, "**", "*", "*.fastq"), recursive=True):
    shutil.move(file, config.PATH)

## Convert FASTQ data to CSV file with all 6 frame translation

In [ ]:
# === CODON TRANSLATION HELPERS ===
def translate(codon):
    codon_dict = {
        "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
        "TCA": "S", "TCG": "S", "TCC": "S", "TCT": "S",
        "TAT": "Y", "TAC": "Y", "TAA": "-", "TAG": "-",
        "TGC": "C", "TGT": "C", "TGA": "-", "TGG": "W",
        "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
        "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
        "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
        "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
        "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
        "ACA": "T", "ACT": "T", "ACC": "T", "ACG": "T",
        "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
        "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
        "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
        "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
        "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
        "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G"
    }
    return codon_dict.get(codon, "")

def complement(base):
    complement_dict = {"A": "T", "T": "A", "C": "G", "G": "C"}
    return complement_dict.get(base, base)

def reverse_complement(sequence):
    return ''.join(complement(base) for base in sequence[::-1])

def translate_sequence(sequence, frame=0):
    return ''.join(translate(sequence[i:i+3]) for i in range(frame, len(sequence) - 2, 3))

# === FASTQ READING & PROCESSING ===
def read_fastq(fastq_path):
    """
    Reads a FASTQ file and returns a list of (ID, Sequence, Quality).
    """
    records = []
    with open(fastq_path, 'r') as fq:
        while True:
            id_line = fq.readline().strip()
            if not id_line:
                break
            seq = fq.readline().strip()
            fq.readline()  # skip '+'
            qual = fq.readline().strip()
            seq_id = id_line[1:] if id_line.startswith('@') else id_line
            records.append((seq_id, seq, qual))
    return records

def phred_score(qual_str):
    """
    Convert ASCII-encoded Phred33 string to average quality score.
    """
    scores = [ord(c) - 33 for c in qual_str]
    return np.mean(scores) if scores else np.nan

def process_fastq(fastq_path, translate_flag=True):
    """
    Process FASTQ --> DataFrame with translation & quality metrics.
    """
    records = read_fastq(fastq_path)
    if not records:
        print(f"No reads found in {fastq_path}")
        return None

    df = pd.DataFrame(records, columns=["ID", "Sequence", "Quality"])
    df["Seq_Length"] = df["Sequence"].str.len()
    df["Avg_Quality"] = df["Quality"].apply(phred_score)

    # Remove ambiguous sequences
    df = df[~df["Sequence"].str.contains('N')]

    if translate_flag:
        print("Translating sequences (6 frames)...")
        for i in range(1, 4):
            df[f'Frame_fwd_{i}'] = df["Sequence"].apply(lambda s: translate_sequence(s, frame=i-1))
            df[f'Frame_rev_{i}'] = df["Sequence"].apply(lambda s: translate_sequence(reverse_complement(s), frame=i-1))
    else:
        print("Skipping translation (--no-translate enabled)")

    return df

def save_to_csv(df, output_path):
    """
    Save DataFrame to plain CSV.
    """
    df.to_csv(output_path + ".csv", index=False)
    print(f"Saved to {output_path}.csv")

def batch_convert_fastq(input_folder, translate_flag=True):
    """
    Convert all FASTQ files in a folder.
    """
    fastq_files = glob.glob(os.path.join(input_folder, "*.fastq")) + \
                  glob.glob(os.path.join(input_folder, "*.fq"))

    if not fastq_files:
        print("No FASTQ files found in the folder.")
        return

    for fastq_file in fastq_files:
        print(f"Processing {fastq_file}...")
        df = process_fastq(fastq_file, translate_flag)
        if df is not None:
            output_path = os.path.splitext(fastq_file)[0]
            save_to_csv(df, output_path)

translate_flag = config.TRANSLATE_FLAG

if os.path.isdir(config.PATH):
    batch_convert_fastq(config.PATH, translate_flag)
else:
    df = process_fastq(config.PATH, translate_flag)
    if df is not None:
        output_path = os.path.splitext(config.PATH)[0]
        save_to_csv(df, output_path)


## Look for motifs or flanking regions

In [ ]:
def find_hits(csv_path, motif=None, flanked_by=None, output_dir=None):
    """
    Find reads containing a given motif or a loop flanked by specified sequences.
    
    Args:
        csv_path: Input CSV (from translated FASTQ data)
        motif: Motif string to search for (optional)
        flanked_by: Tuple or list [left_flank, right_flank] (optional)
        output_dir: Optional path for output folder
    """
    base_dir = os.path.dirname(csv_path)
    base_name = os.path.splitext(os.path.basename(csv_path))[0]

    # --- Output directory setup ---
    if output_dir is None:
        output_dir = os.path.join(base_dir, "outputs")
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nAnalyzing {os.path.basename(csv_path)}...")
    df = pd.read_csv(csv_path)
    frame_cols = [col for col in df.columns if col.startswith("Frame_")]
    matches = []

    # Build regex for loop search or motif search
    if flanked_by and len(flanked_by) == 2:
        left, right = flanked_by
        regex = re.compile(f"{left}([A-Z]+?){right}")
        search_mode = "flanked"
        print(f"Searching for loops flanked by '{left}' and '{right}'")
    elif motif:
        regex = re.compile(re.escape(motif))
        search_mode = "motif"
        print(f"Searching for motif '{motif}'")
    else:
        print("Must provide either --motif or --flanked_by arguments.")
        return

    # Iterate through all frames and sequences
    for _, row in df.iterrows():
        for frame_col in frame_cols:
            aa_seq = str(row[frame_col])

            # Flanked loop search
            if search_mode == "flanked":
                for match in regex.finditer(aa_seq):
                    loop_seq = match.group(1)
                    matches.append({
                        "Read_ID": row["ID"],
                        "Frame": frame_col,
                        "Sequence": row["Sequence"],
                        "Translation": aa_seq,
                        "Flank_Left": left,
                        "Loop_Seq": loop_seq,
                        "Flank_Right": right,
                        "Loop_Length": len(loop_seq),
                        "Avg_Quality": row.get("Avg_Quality", None),
                        "Seq_Length": row.get("Seq_Length", None)
                    })

            # Simple motif search
            elif search_mode == "motif" and regex.search(aa_seq):
                matches.append({
                    "Read_ID": row["ID"],
                    "Frame": frame_col,
                    "Sequence": row["Sequence"],
                    "Translation": aa_seq,
                    "Motif": motif,
                    "Avg_Quality": row.get("Avg_Quality", None),
                    "Seq_Length": row.get("Seq_Length", None)
                })
                break  # Only first matching frame is relevant

    if not matches:
        print(f"No matches found in {csv_path}")
        return None

    matches_df = pd.DataFrame(matches)

    # --- Create summaries ---
    if search_mode == "flanked":
        # Summary 1: Frame-level stats
        frame_summary = (
            matches_df.groupby("Frame")
            .agg(
                Count=("Read_ID", "count"),
                Mean_Quality=("Avg_Quality", "mean"),
                Mean_Seq_Length=("Seq_Length", "mean")
            )
            .reset_index()
        )

        # Summary 2: Unique loop sequences
        loop_summary = (
            matches_df.groupby("Loop_Seq")
            .agg(
                Count=("Read_ID", "count"),
                Mean_Quality=("Avg_Quality", "mean"),
                Mean_Seq_Length=("Seq_Length", "mean"),
                Loop_Length=("Loop_Length", "first")
            )
            .reset_index()
            .sort_values(by="Count", ascending=False)
        )

        # Save outputs inside the output folder
        matches_csv = os.path.join(output_dir, f"{base_name}_loops_{left}_{right}.csv")
        frame_summary_csv = os.path.join(output_dir, f"{base_name}_loops_{left}_{right}_frame_summary.csv")
        loop_summary_csv = os.path.join(output_dir, f"{base_name}_loops_{left}_{right}_loop_summary.csv")

        matches_df.to_csv(matches_csv, index=False)
        frame_summary.to_csv(frame_summary_csv, index=False)
        loop_summary.to_csv(loop_summary_csv, index=False)

        print(f"Matches saved to {matches_csv}")
        print(f"Frame summary saved to {frame_summary_csv}")
        print(f"Loop summary saved to {loop_summary_csv}")

    else:
        # Motif mode summary
        motif_summary = (
            matches_df.groupby("Frame")
            .agg(
                Count=("Read_ID", "count"),
                Mean_Quality=("Avg_Quality", "mean"),
                Mean_Seq_Length=("Seq_Length", "mean")
            )
            .reset_index()
        )

        matches_csv = os.path.join(output_dir, f"{base_name}_{motif}_hits.csv")
        summary_csv = os.path.join(output_dir, f"{base_name}_{motif}_summary.csv")

        matches_df.to_csv(matches_csv, index=False)
        motif_summary.to_csv(summary_csv, index=False)

        print(f"Matches saved to {matches_csv}")
        print(f"Summary saved to {summary_csv}")

    return matches_df


def batch_find_hits(input_folder, motif=None, flanked_by=None):
    """
    Apply motif or loop search to all CSVs in a folder.
    """
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    if not csv_files:
        print("No CSV files found in the folder.")
        return

    # Create a single output folder in batch mode
    output_dir = os.path.join(input_folder, "outputs")
    os.makedirs(output_dir, exist_ok=True)

    for csv_file in csv_files:
        find_hits(csv_file, motif=motif, flanked_by=flanked_by, output_dir=output_dir)

if os.path.isdir(config.PATH):
    batch_find_hits(config.PATH, motif=config.MOTIF, flanked_by=config.FLANKED_BY)
else:
    find_hits(config.PATH, motif=config.MOTIF, flanked_by=config.FLANKED_BY, output_dir=None)

## Compare sequences across NGS runs

Update the indicies of `FILE_1` and `FILE_2` to compare the sequences between different files

In [ ]:
summary_files = sorted(glob.glob(os.path.join(config.PATH, "outputs", "*loop_summary.csv")))
FILE_1 = summary_files[0]
FILE_2 = summary_files[2]

print(FILE_1)
print(FILE_2)

COLUMN_TO_COMPARE = 'Loop_Seq'
HEADER_SETTING = 0  # 0 means the first row is a header. None if no header

# --- Header Check ---
if isinstance(COLUMN_TO_COMPARE, str) and HEADER_SETTING is None:
    print(f"Error: You specified a column name ('{COLUMN_TO_COMPARE}')")
    print("but set HEADER_SETTING to None. Please set HEADER_SETTING = 0.")
    raise SystemExit

# --- Logic ---
try:
    # Read *only* the specified column from each file
    # .squeeze() converts the single-column DataFrame into a Series
    col1 = pd.read_csv(
        FILE_1, 
        header=HEADER_SETTING, 
        usecols=[COLUMN_TO_COMPARE]
    ).squeeze()
    
    col2 = pd.read_csv(
        FILE_2, 
        header=HEADER_SETTING, 
        usecols=[COLUMN_TO_COMPARE]
    ).squeeze()

    # Convert the columns to sets to find unique items
    set1 = set(col1)
    set2 = set(col2)

    # --- Calculations ---
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    # Calculate Jaccard Similarity
    if len(union) == 0:
        jaccard_similarity = 1.0 if len(intersection) == 0 else 0.0
    else:
        jaccard_similarity = len(intersection) / len(union)

    # Find items unique to each file
    only_in_file1 = set1.difference(set2)
    only_in_file2 = set2.difference(set1)

    # --- Results ---
    print(f"--- Column Similarity Report for '{COLUMN_TO_COMPARE}' ---")
    print(f"\nJaccard Similarity Score: {jaccard_similarity:.4f}")
    print(f" (1.0 = identical sets, 0.0 = no overlap)")

    print(f"\n--- Counts ---")
    print(f"Total unique items in File 1: {len(set1)}")
    print(f"Total unique items in File 2: {len(set2)}")
    print(f"Items in *both* files (Intersection): {len(intersection)}")
    print(f"Total unique items in *all* files (Union): {len(union)}")
    print(f"Items *only* in File 1: {len(only_in_file1)}")
    print(f"Items *only* in File 2: {len(only_in_file2)}")

except FileNotFoundError as e:
    print(f"Error: {e}")
except pd.errors.EmptyDataError:
    print("Error: One of the files is empty.")
except ValueError as e:
    # This often happens if the COLUMN_TO_COMPARE doesn't exist
    print(f"ValueError: {e}")
    print(f"Check if '{COLUMN_TO_COMPARE}' is the correct column name/index.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")